# DataCo Supply Chain Analysis

**What this project does:**  
We have 180,519 orders from a global supply chain company. This notebook cleans the data, fixes errors in it, runs SQL queries to find business insights, calculates inventory numbers, and finally exports a clean file for the Power BI dashboard.

**Dataset:** DataCoSupplyChainDataset.csv  
**Author:** Tanish Kumain  
**Tools used:** Python, Pandas, SQL (SQLite), Power BI

---

**Sections in this notebook:**
1. Load the data
2. Clean the data
3. Create new useful columns
4. Fix data quality problems
5. Answer business questions using SQL
6. Inventory analysis (Safety Stock, EOQ, ABC)
7. Export clean data for Power BI

---
## Section 1 — Load the Data

First we import the libraries we need, then load the CSV file.

In [ ]:
# Import all the tools we need
import pandas as pd        # for working with tables of data
import numpy as np         # for math operations
import matplotlib.pyplot as plt   # for charts
import seaborn as sns      # for nicer charts
import sqlite3             # for running SQL queries
import math                # needed for the EOQ square root formula
import warnings
warnings.filterwarnings('ignore')  # hide unnecessary warning messages

print('All libraries loaded successfully.')

In [ ]:
# Load the raw dataset
# encoding='latin1' is needed because the file has some special characters
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')

print(f'Dataset loaded: {df.shape[0]:,} rows and {df.shape[1]} columns')
df.head(3)

---
## Section 2 — Clean the Data

The raw dataset has 53 columns. Many of them are useless for analysis:
- Personal info like email and password (we should never use these)
- Columns that are completely empty or have too many missing values
- Duplicate columns that say the exact same thing

We will keep only the 30 columns that actually matter for our analysis.

In [ ]:
# Check which columns have missing values
missing = df.isnull().sum()

print('Columns that have missing data:')
print(missing[missing > 0])

In [ ]:
# Choose which columns to keep
# We are dropping:
#   - Customer email, password, street address (personal info - not needed)
#   - Product Description (almost completely empty)
#   - Order Zipcode (86% missing values)
#   - Product Image (just a URL link - useless for analysis)
#   - benefit_per_order (exact copy of Order Profit Per Order)
#   - sales_per_customer (exact copy of Order Item Total)

columns_to_keep = [
    # Payment and shipping info
    'Type',                            # How the customer paid (Debit, Transfer, Cash)
    'Days for shipping (real)',        # How many days the order actually took to ship
    'Days for shipment (scheduled)',   # How many days it was supposed to take
    'Delivery Status',                 # Was it on time, late, cancelled?
    'Late_delivery_risk',              # System flag: 1 = late risk, 0 = safe
    'Shipping Mode',                   # First Class / Second Class / Standard / Same Day

    # Customer info
    'Customer Segment',   # Consumer, Corporate, or Home Office
    'Customer City',
    'Customer Country',

    # Where the order was going
    'Market',             # Europe, LATAM, Pacific Asia, USCA, Africa
    'Order Region',       # More specific area within the market (23 regions total)
    'Order Country',
    'Order City',
    'Latitude',           # Needed for map visuals in Power BI
    'Longitude',

    # Product info
    'Category Name',
    'Department Name',
    'Product Name',
    'Product Price',

    # Order identifiers and status
    'Order Id',
    'Order Item Id',
    'Order Status',       # COMPLETE, PENDING, CANCELED, SUSPECTED_FRAUD etc.
    'order date (DateOrders)',
    'shipping date (DateOrders)',

    # Financial info
    'Sales',                      # Total price before discount
    'Order Item Total',           # Price after discount (what customer actually paid)
    'Order Item Discount',        # How much discount was given in dollars
    'Order Item Discount Rate',   # Discount as a percentage
    'Order Item Quantity',        # How many units were ordered
    'Order Item Product Price',   # Unit price of the product
    'Order Item Profit Ratio',    # Profit as a ratio per item
    'Order Profit Per Order',     # Net profit on this order
]

df_clean = df[columns_to_keep].copy()

print(f'Kept {df_clean.shape[1]} columns out of the original {df.shape[1]}')
print(f'Dropped {df.shape[1] - df_clean.shape[1]} unnecessary columns')
df_clean.info()

In [ ]:
# Fix the date columns
# The dates are currently stored as plain text strings like "1/31/2015 22:56"
# We need to convert them to proper date format so Python understands them as dates

df_clean['order_date']    = pd.to_datetime(df_clean['order date (DateOrders)'],    errors='coerce')
df_clean['shipping_date'] = pd.to_datetime(df_clean['shipping date (DateOrders)'], errors='coerce')

# Remove the original messy-named date columns since we have clean ones now
df_clean.drop(columns=['order date (DateOrders)', 'shipping date (DateOrders)'], inplace=True)

# Check how many dates failed to parse (should be 0)
print('Dates that could not be parsed (should be 0):')
print(df_clean[['order_date', 'shipping_date']].isnull().sum())
print(f'\nDate range in our data: {df_clean["order_date"].min().date()} to {df_clean["order_date"].max().date()}')

In [ ]:
# Check for any remaining missing values after our column selection
missing_after = df_clean.isnull().sum()
missing_after = missing_after[missing_after > 0]

if len(missing_after) == 0:
    print('No missing values found. All columns are fully populated.')
else:
    print('Columns still missing data:')
    print(missing_after)

In [ ]:
# Check for duplicate rows (same exact data repeated)
duplicate_count = df_clean.duplicated().sum()
print(f'Number of duplicate rows: {duplicate_count}')

if duplicate_count > 0:
    df_clean = df_clean.drop_duplicates(keep='first')
    print(f'Duplicates removed. Dataset now has {df_clean.shape[0]:,} rows.')
else:
    print('No duplicates found.')

---
## Section 3 — Create New Useful Columns

The raw data does not have columns like "how many days late was this order?" or "what month was this ordered in?"  
We create these ourselves from the existing data. This is called **feature engineering**.

In [ ]:
# How many days late (or early) was each order?
# Positive number = late, 0 = exactly on time, negative = arrived early
df_clean['shipping_delay_days'] = (
    df_clean['Days for shipping (real)'] - df_clean['Days for shipment (scheduled)']
)

# Break the order date into parts for time-based analysis
df_clean['order_year']        = df_clean['order_date'].dt.year
df_clean['order_month']       = df_clean['order_date'].dt.month
df_clean['order_quarter']     = df_clean['order_date'].dt.quarter
df_clean['order_year_month']  = df_clean['order_date'].dt.to_period('M').astype(str)  # e.g. "2015-03"
df_clean['order_day_of_week'] = df_clean['order_date'].dt.day_name()                  # e.g. "Monday"

# Revenue at risk = sales value of orders that arrived late
# If an order was late, its full sales value is "at risk" of return or customer complaint
df_clean['revenue_at_risk'] = np.where(
    df_clean['shipping_delay_days'] > 0,
    df_clean['Sales'],
    0
)

# Flag fraud and cancelled orders as 1 or 0 for easy counting later
df_clean['is_fraud']     = (df_clean['Order Status'] == 'SUSPECTED_FRAUD').astype(int)
df_clean['is_cancelled'] = (df_clean['Order Status'] == 'CANCELED').astype(int)

print('New columns created successfully.')
print('\nQuick stats on the new columns:')
print(df_clean[['shipping_delay_days', 'revenue_at_risk', 'is_fraud', 'is_cancelled']].describe().round(2))

---
## Section 4 — Fix Data Quality Problems

After exploring the data we found several problems:
1. **4,423 wrong risk flags** — The system said these orders were safe but they were actually late
2. **Two status labels for the same thing** — PENDING and PENDING_PAYMENT are identical
3. **Cancelled orders show full revenue** — A cancelled order should show $0 revenue
4. **Too many decimal places** — Numbers like 294.980011 should be 294.98

We fix all of these below.

In [ ]:
# Check 1: Are there any orders with negative shipping days? (impossible in reality)
negative_days = (df_clean['Days for shipping (real)'] < 0).sum()
print(f'Orders with impossible negative shipping days: {negative_days}')

# Check 2: How many risk flags does the system have wrong?
# Wrong means: system said 0 (safe) but shipping_delay_days > 0 (actually late)
wrong_flags = df_clean[
    (df_clean['Late_delivery_risk'] == 0) & (df_clean['shipping_delay_days'] > 0)
].shape[0]
print(f'\nWrong late delivery flags (system said safe, but was actually late): {wrong_flags:,}')

# Check 3: Does the math add up? (Sales - Discount should equal Order Item Total)
expected = df_clean['Sales'] - df_clean['Order Item Discount']
pricing_errors = (abs(df_clean['Order Item Total'] - expected) > 0.01).sum()
print(f'\nRows where Sales - Discount does not equal Order Item Total: {pricing_errors:,}')

# Check 4: What are all the order statuses and how many of each?
print(f'\nAll order statuses in the dataset:')
print(df_clean['Order Status'].value_counts())

print(f'\nFraud rate: {df_clean["is_fraud"].mean()*100:.1f}% of all orders')
print(f'Cancellation rate: {df_clean["is_cancelled"].mean()*100:.1f}% of all orders')

In [ ]:
# Drill deeper into the wrong flags — which shipping modes have the most errors?
wrong_flag_rows = df_clean[
    (df_clean['Late_delivery_risk'] == 0) & (df_clean['shipping_delay_days'] > 0)
]

print('Wrong flags by shipping mode:')
print(wrong_flag_rows['Shipping Mode'].value_counts())

print('\nWrong flags by market:')
print(wrong_flag_rows['Market'].value_counts())

In [ ]:
# FIX 1: Correct the wrong late delivery flags
# Simple rule: if shipping_delay_days > 0 then it IS late (1), otherwise it is NOT late (0)
df_clean.loc[df_clean['shipping_delay_days'] > 0,  'Late_delivery_risk'] = 1
df_clean.loc[df_clean['shipping_delay_days'] <= 0, 'Late_delivery_risk'] = 0

total_late = df_clean['Late_delivery_risk'].sum()
true_rate  = total_late / len(df_clean) * 100
print(f'Flags corrected.')
print(f'True late delivery rate: {true_rate:.1f}%')
print(f'(The system was reporting ~40% — the real number is {true_rate:.1f}%)')

# FIX 2: Merge PENDING_PAYMENT into PENDING
# We checked: both statuses have identical sales, profit, and delivery patterns
# They represent the same thing — an order placed but not yet processed
df_clean['Order Status'] = df_clean['Order Status'].replace('PENDING_PAYMENT', 'PENDING')
print(f'\nOrder statuses after merging PENDING_PAYMENT into PENDING:')
print(df_clean['Order Status'].value_counts())

# FIX 3: Set cancelled orders to $0 revenue with a small negative profit
# Business logic: a cancelled order earns nothing, but costs money to process
# We estimate the cancellation handling cost at 8% of the original order value
CANCELLATION_COST = 0.08
cancelled = df_clean['Order Status'] == 'CANCELED'

original_sales = df_clean.loc[cancelled, 'Sales'].copy()

df_clean.loc[cancelled, 'Sales']                   = 0.00
df_clean.loc[cancelled, 'Order Item Total']         = 0.00
df_clean.loc[cancelled, 'Order Item Discount']      = 0.00
df_clean.loc[cancelled, 'Order Item Discount Rate'] = 0.00
df_clean.loc[cancelled, 'Order Profit Per Order']   = -(original_sales * CANCELLATION_COST).round(2)
df_clean.loc[cancelled, 'Order Item Profit Ratio']  = -CANCELLATION_COST
df_clean.loc[cancelled, 'Late_delivery_risk']       = 0
df_clean.loc[cancelled, 'shipping_delay_days']      = 0
df_clean.loc[cancelled, 'revenue_at_risk']          = 0.00

print(f'\n{cancelled.sum():,} cancelled orders fixed (sales set to $0, profit set to negative)')

# FIX 4: Round all decimal numbers to maximum 2 decimal places
# This removes ugly numbers like 294.980011 and keeps things like 294.98
float_columns = df_clean.select_dtypes(include=['float64', 'float32']).columns
df_clean[float_columns] = df_clean[float_columns].round(2)
print(f'\nRounded {len(float_columns)} columns to 2 decimal places')

In [ ]:
# Detect unusually high sales values (outliers)
# We use the IQR method: anything above Q3 + 1.5×IQR is an outlier
Q1 = df_clean['Sales'].quantile(0.25)
Q3 = df_clean['Sales'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR

df_clean['is_sales_outlier'] = (df_clean['Sales'] > upper_limit).astype(int)
outlier_rows = df_clean[df_clean['is_sales_outlier'] == 1]

print(f'Sales outlier threshold: ${upper_limit:.2f}')
print(f'Number of high-value outlier rows: {len(outlier_rows):,}')
print(f'Average outlier sale value: ${outlier_rows["Sales"].mean():.2f}')
print('\nProducts appearing most in outliers:')
print(outlier_rows['Product Name'].value_counts().head(5))

In [ ]:
# Final summary of our cleaned dataset
print('=== Final Data Quality Summary ===')
print(f'Total rows:            {len(df_clean):,}')
print(f'Total columns:         {df_clean.shape[1]}')
print(f'Missing values:        {df_clean.isnull().sum().sum()}')
print(f'Date range:            {df_clean["order_date"].min().date()} to {df_clean["order_date"].max().date()}')
print(f'Late delivery rate:    {df_clean["Late_delivery_risk"].mean()*100:.1f}%')
print(f'Fraud rate:            {df_clean["is_fraud"].mean()*100:.1f}%')
print(f'Cancellation rate:     {df_clean["is_cancelled"].mean()*100:.1f}%')
print('==================================')

---
## Section 5 — SQL Analysis

Now that the data is clean, we load it into an in-memory SQL database and run business intelligence queries.

SQL is easier than pandas for complex grouping and aggregation questions like:  
*"What is the revenue, profit, and discount drain for each market?"*

We have **12 queries** covering:
- Revenue and profit by market, category, and segment
- Payment methods and fraud patterns
- Shipping performance and late delivery rates
- Seasonal and day-of-week demand patterns

In [ ]:
# Load the cleaned data into an in-memory SQLite database
# We also register a SQRT function because SQLite does not have one built in
# (we need SQRT later for the EOQ formula)
conn = sqlite3.connect(':memory:')
conn.create_function('SQRT', 1, math.sqrt)

df_clean.to_sql('orders', conn, index=False, if_exists='replace')

print(f'Loaded {len(df_clean):,} rows into SQLite.')
print('Table name: orders')

In [ ]:
# Query 1: Which market makes the most revenue and profit?
print('--- Revenue and Profit by Market ---')

query = """
SELECT
    Market,
    COUNT("Order Id")                                              AS Total_Orders,
    ROUND(SUM(Sales), 2)                                          AS Total_Revenue,
    ROUND(SUM("Order Item Discount"), 2)                         AS Total_Discounts,
    ROUND(SUM("Order Profit Per Order"), 2)                      AS Total_Profit,
    ROUND(AVG("Order Item Profit Ratio") * 100, 1)               AS Avg_Profit_Ratio_Pct,
    ROUND((SUM("Order Item Discount") / SUM(Sales)) * 100, 2)   AS Discount_Drain_Pct
FROM orders
GROUP BY Market
ORDER BY Total_Revenue DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
# Query 2: Which product categories make the most revenue?
print('--- Top 10 Categories by Revenue ---')

query = """
SELECT
    "Category Name",
    COUNT("Order Id")                              AS Total_Orders,
    ROUND(SUM(Sales), 2)                          AS Total_Revenue,
    ROUND(SUM("Order Profit Per Order"), 2)       AS Total_Profit,
    ROUND(AVG("Order Item Profit Ratio") * 100, 1) AS Avg_Profit_Ratio_Pct
FROM orders
GROUP BY "Category Name"
ORDER BY Total_Revenue DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

In [ ]:
# Query 3: Which customer type (segment) is most profitable?
# We have Consumer, Corporate, and Home Office customers
print('--- Revenue and Profit by Customer Segment ---')

query = """
SELECT
    "Customer Segment",
    COUNT("Order Id")                                             AS Total_Orders,
    ROUND(SUM(Sales), 2)                                         AS Total_Revenue,
    ROUND(SUM("Order Profit Per Order"), 2)                     AS Total_Profit,
    ROUND(AVG("Order Item Profit Ratio") * 100, 1)              AS Avg_Profit_Ratio_Pct,
    ROUND(SUM("Order Item Discount"), 2)                        AS Total_Discounts,
    ROUND((SUM("Order Item Discount") / SUM(Sales)) * 100, 2)  AS Discount_Drain_Pct
FROM orders
GROUP BY "Customer Segment"
ORDER BY Total_Revenue DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
# Query 4: How do customers pay and does it affect revenue?
print('--- Revenue by Payment Method ---')

query = """
SELECT
    Type                    AS Payment_Method,
    COUNT("Order Id")       AS Total_Orders,
    ROUND(SUM(Sales), 2)   AS Total_Revenue,
    ROUND(
        COUNT("Order Id") * 100.0 / (SELECT COUNT(*) FROM orders),
        1
    )                       AS Order_Share_Pct
FROM orders
GROUP BY Type
ORDER BY Total_Revenue DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
# Query 5: Where is fraud happening and which segment is affected most?
print('--- Fraud and Cancellation by Market and Customer Segment ---')

query = """
SELECT
    Market,
    "Customer Segment",
    COUNT("Order Id")                                                 AS Total_Orders,
    SUM(is_fraud)                                                     AS Fraud_Orders,
    ROUND(SUM(is_fraud) * 100.0 / COUNT("Order Id"), 1)              AS Fraud_Pct,
    SUM(is_cancelled)                                                 AS Cancelled_Orders,
    ROUND(SUM(is_cancelled) * 100.0 / COUNT("Order Id"), 1)          AS Cancel_Pct,
    ROUND(SUM(CASE WHEN is_fraud = 1 THEN Sales ELSE 0 END), 2)      AS Fraud_Revenue_Exposure
FROM orders
GROUP BY Market, "Customer Segment"
ORDER BY Fraud_Pct DESC
LIMIT 15;
"""

pd.read_sql_query(query, conn)

In [ ]:
# Query 6: Which shipping mode has the most delays and puts the most revenue at risk?
print('--- Shipping Mode Performance ---')

query = """
SELECT
    "Shipping Mode",
    COUNT("Order Id")                                                         AS Total_Orders,
    SUM(CASE WHEN shipping_delay_days > 0 THEN 1 ELSE 0 END)                 AS Delayed_Orders,
    ROUND(
        SUM(CASE WHEN shipping_delay_days > 0 THEN 1 ELSE 0 END) * 100.0
        / COUNT("Order Id"),
        1
    )                                                                          AS Delay_Rate_Pct,
    ROUND(SUM(revenue_at_risk), 2)                                            AS Revenue_At_Risk,
    ROUND(AVG(shipping_delay_days), 2)                                        AS Avg_Delay_Days
FROM orders
GROUP BY "Shipping Mode"
ORDER BY Delay_Rate_Pct DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
# Query 7: Which specific regions have the worst delivery performance?
# We use Order Region (23 regions) instead of just Market (5 markets)
# for a more granular view
print('--- Late Delivery Rate by Region (Top 20 worst) ---')

query = """
SELECT
    Market,
    "Order Region",
    COUNT("Order Id")                                               AS Total_Orders,
    SUM(Late_delivery_risk)                                         AS Late_Orders,
    ROUND(SUM(Late_delivery_risk) * 100.0 / COUNT("Order Id"), 1)  AS Late_Pct,
    ROUND(SUM(revenue_at_risk), 2)                                  AS Revenue_At_Risk
FROM orders
GROUP BY Market, "Order Region"
ORDER BY Late_Pct DESC
LIMIT 20;
"""

pd.read_sql_query(query, conn)

In [ ]:
# Query 8: Which specific products have the most money stuck in delayed shipments?
print('--- Top Products with Revenue at Risk (from delays) ---')

query = """
SELECT
    "Product Name",
    "Shipping Mode",
    COUNT("Order Id")               AS Delayed_Orders,
    ROUND(SUM(revenue_at_risk), 2)  AS Revenue_At_Risk
FROM orders
WHERE shipping_delay_days > 0
GROUP BY "Product Name", "Shipping Mode"
ORDER BY Revenue_At_Risk DESC
LIMIT 15;
"""

pd.read_sql_query(query, conn)

In [ ]:
# Query 9: Which days of the week get the most orders?
# This helps with warehouse staffing and replenishment scheduling
print('--- Order Volume by Day of Week ---')

query = """
SELECT
    order_day_of_week                                               AS Day_of_Week,
    COUNT("Order Id")                                              AS Total_Orders,
    ROUND(SUM(Sales), 2)                                          AS Total_Revenue,
    ROUND(SUM(Late_delivery_risk) * 100.0 / COUNT("Order Id"), 1) AS Late_Delivery_Pct
FROM orders
GROUP BY order_day_of_week
ORDER BY Total_Orders DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
# Query 10: Does demand go up or down depending on the month?
# This helps plan inventory levels ahead of busy and quiet periods
print('--- Monthly Demand Seasonality ---')

query = """
SELECT
    STRFTIME('%m', order_date)  AS Month_Number,
    COUNT("Order Id")           AS Total_Orders,
    ROUND(SUM(Sales), 2)        AS Total_Revenue,
    ROUND(AVG(Sales), 2)        AS Avg_Order_Value
FROM orders
GROUP BY Month_Number
ORDER BY Month_Number;
"""

pd.read_sql_query(query, conn)

In [ ]:
# Query 11: How did each market perform quarter by quarter?
# This shows which markets are growing and which are shrinking over time
print('--- Quarterly Revenue by Market (2015-2016) ---')

query = """
SELECT
    order_year                              AS Year,
    order_quarter                           AS Quarter,
    Market,
    COUNT("Order Id")                      AS Total_Orders,
    ROUND(SUM(Sales), 2)                   AS Quarterly_Revenue,
    ROUND(SUM("Order Profit Per Order"), 2) AS Quarterly_Profit
FROM orders
WHERE order_year IN (2015, 2016)
GROUP BY Year, Quarter, Market
ORDER BY Year, Quarter, Market;
"""

pd.read_sql_query(query, conn)

In [ ]:
# Query 12: How much money do we actually keep after discounts?
# This is the cash flow picture: Gross Revenue → Discounts → Net Profit
print('--- Cash Flow by Market (Revenue vs Discounts vs Profit) ---')

query = """
SELECT
    Market,
    COUNT("Order Id")                                               AS Total_Orders,
    ROUND(SUM(Sales), 2)                                           AS Gross_Revenue,
    ROUND(SUM("Order Item Discount"), 2)                          AS Cash_Lost_To_Discounts,
    ROUND(SUM("Order Profit Per Order"), 2)                       AS Net_Profit,
    ROUND((SUM("Order Item Discount") / SUM(Sales)) * 100, 2)    AS Discount_Drain_Pct,
    ROUND((SUM("Order Profit Per Order") / SUM(Sales)) * 100, 2) AS Net_Margin_Pct
FROM orders
GROUP BY Market
ORDER BY Gross_Revenue DESC;
"""

pd.read_sql_query(query, conn)

---
## Section 6 — Inventory Analysis

This section answers three inventory management questions:

1. **Safety Stock** — How much extra stock should we keep to avoid running out?
2. **EOQ (Economic Order Quantity)** — What is the most cost-efficient order size for each product?
3. **ABC Classification** — Which products are our most valuable (A), middle-tier (B), and low-value (C)?

These calculations help the supply chain team decide what to stock more of and what to reduce.

In [ ]:
# First: understand how consistent our lead times are for high-volume products
# Lead time = how long it actually takes for an order to ship
# High variance = unpredictable supply = need more safety stock
print('--- Lead Time Analysis for High-Volume Products ---')

query = """
SELECT
    "Product Name",
    COUNT("Order Id")                                          AS Total_Orders,
    ROUND(AVG("Days for shipping (real)"), 2)                AS Avg_Lead_Time,
    MAX("Days for shipping (real)")                          AS Max_Lead_Time,
    (MAX("Days for shipping (real)") - AVG("Days for shipping (real)")) AS Lead_Time_Variance
FROM orders
GROUP BY "Product Name"
HAVING Total_Orders > 5000
ORDER BY Total_Orders DESC;
"""

pd.read_sql_query(query, conn)

In [ ]:
# Safety Stock calculation
# Formula: Safety Stock = (Max Daily Demand × Max Lead Time) - (Avg Daily Demand × Avg Lead Time)
# This tells us: in the worst case scenario (highest demand, longest lead time)
# how much extra stock do we need compared to the average scenario?
print('--- Safety Stock Buffer for High-Volume Products ---')

query = """
WITH DailySales AS (
    -- Calculate how many units were sold each day per product
    SELECT
        "Product Name",
        DATE(order_date)            AS Sales_Day,
        SUM("Order Item Quantity")  AS Units_Sold
    FROM orders
    GROUP BY "Product Name", DATE(order_date)
),
ProductInfo AS (
    -- Get lead time info for each high-volume product
    SELECT
        "Product Name",
        COUNT("Order Id")                           AS Total_Orders,
        ROUND(AVG("Days for shipping (real)"), 2)  AS Avg_Lead_Time,
        MAX("Days for shipping (real)")            AS Max_Lead_Time
    FROM orders
    GROUP BY "Product Name"
    HAVING Total_Orders > 5000
)
SELECT
    p."Product Name",
    p.Total_Orders,
    ROUND(AVG(d.Units_Sold), 2)  AS Avg_Daily_Demand,
    MAX(d.Units_Sold)            AS Max_Daily_Demand,
    p.Avg_Lead_Time,
    p.Max_Lead_Time,
    ROUND(
        (MAX(d.Units_Sold) * p.Max_Lead_Time) - (AVG(d.Units_Sold) * p.Avg_Lead_Time),
        0
    ) AS Safety_Stock_Buffer
FROM ProductInfo p
JOIN DailySales d ON p."Product Name" = d."Product Name"
GROUP BY p."Product Name"
ORDER BY Safety_Stock_Buffer DESC;
"""

safety_stock_df = pd.read_sql_query(query, conn)
safety_stock_df

In [ ]:
# Before calculating EOQ we need to know what categories exist in the data
# This prevents us from hardcoding category names that might be spelled differently
all_categories = pd.read_sql_query(
    'SELECT DISTINCT "Category Name" FROM orders ORDER BY 1', conn
)['Category Name'].tolist()

print('All product categories in our dataset:')
for cat in all_categories:
    print(f'  - {cat}')

# Define which categories cost more to hold in the warehouse
# (large/heavy items like gym equipment and camping gear cost more per unit to store)
HIGH_HOLDING_COST_CATEGORIES = (
    "'Cardio Equipment', 'Fitness Accessories', 'Strength Training', "
    "'Camping & Hiking', 'Water Sports'"
)

In [ ]:
# EOQ = Economic Order Quantity
# This formula tells us the most cost-efficient quantity to order at a time
# Formula: EOQ = SQRT( (2 × Annual Demand × Ordering Cost) / Holding Cost per Unit )
#
# Assumptions:
# - Ordering cost = $50 per order (fixed cost regardless of size)
# - Holding cost = $4.00/unit for large items, $1.50/unit for small items
print('--- EOQ (Optimal Order Quantity) for Top 30 Products ---')

query = f"""
WITH AnnualDemand AS (
    -- Calculate average annual demand per product
    SELECT
        "Product Name",
        "Category Name",
        SUM("Order Item Quantity") / 2.0 AS Avg_Annual_Demand
    FROM orders
    WHERE order_year_month >= '2015-01' AND order_year_month <= '2016-12'
    GROUP BY "Product Name"
    HAVING SUM("Order Item Quantity") > 0
)
SELECT
    "Product Name",
    "Category Name",
    ROUND(Avg_Annual_Demand, 0) AS Annual_Demand,
    CASE
        WHEN "Category Name" IN ({HIGH_HOLDING_COST_CATEGORIES}) THEN 4.00
        ELSE 1.50
    END AS Holding_Cost_Per_Unit,
    50.00 AS Ordering_Cost,
    ROUND(
        SQRT(
            (2 * Avg_Annual_Demand * 50.00) /
            CASE
                WHEN "Category Name" IN ({HIGH_HOLDING_COST_CATEGORIES}) THEN 4.00
                ELSE 1.50
            END
        ),
        0
    ) AS EOQ
FROM AnnualDemand
WHERE Avg_Annual_Demand >= 100
ORDER BY Avg_Annual_Demand DESC
LIMIT 30;
"""

eoq_df = pd.read_sql_query(query, conn)
eoq_df

In [ ]:
# Inventory Velocity — how fast does each product sell?
# Monthly velocity = total units sold ÷ number of months the product was active
# Low velocity = slow-moving product = tying up warehouse space
print('--- Slowest Moving Products (these clog up the warehouse) ---')

query = """
SELECT
    "Product Name",
    "Category Name",
    SUM("Order Item Quantity")                                     AS Total_Units_Sold,
    COUNT(DISTINCT order_year_month)                               AS Months_Active,
    ROUND(
        SUM("Order Item Quantity") * 1.0 / COUNT(DISTINCT order_year_month),
        1
    )                                                               AS Monthly_Velocity
FROM orders
GROUP BY "Product Name"
ORDER BY Monthly_Velocity ASC
LIMIT 15;
"""

pd.read_sql_query(query, conn)

In [ ]:
# ABC Classification
# Class A = high value, low volume (our premium products — handle with care)
# Class B = middle ground (the bulk of our catalogue)
# Class C = low value, high volume (these often get over-discounted and waste shelf space)
#
# We calculate the thresholds from the actual data instead of guessing fixed numbers
# Top 20% revenue = Class A candidate
# Bottom 40% revenue with high volume = Class C candidate

product_summary = pd.read_sql_query("""
    SELECT
        "Product Name",
        "Category Name",
        ROUND(SUM(Sales), 2)        AS Total_Revenue,
        SUM("Order Item Quantity")  AS Total_Units
    FROM orders
    GROUP BY "Product Name"
""", conn)

# Calculate thresholds from our actual data
revenue_80th_percentile = product_summary['Total_Revenue'].quantile(0.80)
revenue_40th_percentile = product_summary['Total_Revenue'].quantile(0.40)
units_60th_percentile   = product_summary['Total_Units'].quantile(0.60)

print('Thresholds calculated from actual data:')
print(f'  Class A threshold (revenue above): ${revenue_80th_percentile:,.0f}')
print(f'  Class C threshold (revenue below): ${revenue_40th_percentile:,.0f}')
print(f'  High inventory footprint (units above): {units_60th_percentile:,.0f}')

In [ ]:
# Apply the ABC labels to each product
def label_abc(row):
    if row['Total_Revenue'] >= revenue_80th_percentile and row['Total_Units'] <= units_60th_percentile:
        return 'Class A: High Value, Low Inventory'
    elif row['Total_Revenue'] <= revenue_40th_percentile and row['Total_Units'] > units_60th_percentile:
        return 'Class C: Low Value, High Inventory'
    else:
        return 'Class B: Balanced'

product_summary['ABC_Class'] = product_summary.apply(label_abc, axis=1)

print('How many products in each class?')
print(product_summary['ABC_Class'].value_counts())
print()

# Add the ABC class label back to our main dataset
df_clean = df_clean.merge(
    product_summary[['Product Name', 'ABC_Class']],
    on='Product Name',
    how='left'
)

print('Top Class A products (our most valuable):')
class_a = product_summary[product_summary['ABC_Class'] == 'Class A: High Value, Low Inventory']
print(class_a.sort_values('Total_Revenue', ascending=False).head(10).to_string(index=False))

In [ ]:
# Class C discount drain audit
# Class C products have low value but we are still giving them the same discounts
# This is wasting money — we should reduce discounts on Class C products
print('--- Class C Products Getting Too Much Discount ---')

query = """
WITH ProductMetrics AS (
    SELECT
        "Product Name",
        ROUND(SUM(Sales), 2)                          AS Total_Revenue,
        SUM("Order Item Quantity")                    AS Total_Units,
        ROUND(SUM("Order Item Discount"), 2)          AS Total_Discounts,
        ROUND(AVG("Order Item Profit Ratio") * 100, 1) AS Avg_Profit_Ratio_Pct
    FROM orders
    GROUP BY "Product Name"
)
SELECT
    "Product Name",
    Total_Revenue,
    Total_Units,
    Total_Discounts,
    ROUND((Total_Discounts / Total_Revenue) * 100, 2) AS Discount_Drain_Pct,
    Avg_Profit_Ratio_Pct
FROM ProductMetrics
WHERE Total_Revenue < {low} AND Total_Units > {high}
ORDER BY Discount_Drain_Pct DESC
LIMIT 20;
""".format(low=round(revenue_40th_percentile, 2), high=round(units_60th_percentile, 2))

pd.read_sql_query(query, conn)

---
## Section 7 — Export Clean Data for Power BI

We now have a fully cleaned, enriched dataset with:
- All data quality issues fixed
- New columns like `shipping_delay_days`, `revenue_at_risk`, `abc_class`
- Column names reformatted to work cleanly in Power BI (no spaces, all lowercase)

We export this as a CSV file that gets loaded directly into the Power BI dashboard.

In [ ]:
# Reload the enriched data into SQLite so ABC class is included
df_clean.to_sql('orders_final', conn, index=False, if_exists='replace')

print(f'Final dataset ready to export')
print(f'Rows: {df_clean.shape[0]:,}')
print(f'Columns: {df_clean.shape[1]}')

In [ ]:
# Rename all columns to Power BI friendly names
# Power BI has issues with column names that have spaces or special characters
# So we rename everything to lowercase with underscores

rename_map = {
    'Type':                          'payment_method',
    'Days for shipping (real)':      'days_shipping_real',
    'Days for shipment (scheduled)': 'days_shipping_scheduled',
    'Delivery Status':               'delivery_status',
    'Late_delivery_risk':            'late_delivery_risk',
    'Shipping Mode':                 'shipping_mode',
    'Customer Segment':              'customer_segment',
    'Customer City':                 'customer_city',
    'Customer Country':              'customer_country',
    'Market':                        'market',
    'Order Region':                  'order_region',
    'Order Country':                 'order_country',
    'Order City':                    'order_city',
    'Latitude':                      'latitude',
    'Longitude':                     'longitude',
    'Category Name':                 'category_name',
    'Department Name':               'department_name',
    'Product Name':                  'product_name',
    'Product Price':                 'product_price',
    'Order Id':                      'order_id',
    'Order Item Id':                 'order_item_id',
    'Order Status':                  'order_status',
    'Sales':                         'sales',
    'Order Item Total':              'order_item_total',
    'Order Item Discount':           'order_item_discount',
    'Order Item Discount Rate':      'order_item_discount_rate',
    'Order Item Quantity':           'order_item_quantity',
    'Order Item Product Price':      'order_item_product_price',
    'Order Item Profit Ratio':       'order_item_profit_ratio',
    'Order Profit Per Order':        'order_profit_per_order',
    'ABC_Class':                     'abc_class',
}

export_df = df_clean.rename(columns=rename_map)

# Convert the year-month period to a plain string (required for CSV)
if 'order_year_month' in export_df.columns:
    export_df['order_year_month'] = export_df['order_year_month'].astype(str)

# One final round of all decimal numbers to 2 decimal places
float_cols = export_df.select_dtypes(include=['float64', 'float32']).columns
export_df[float_cols] = export_df[float_cols].round(2)

print(f'Columns renamed. Final shape: {export_df.shape}')

# Sanity check: make sure cancelled orders are correct
print('\nQuick check — cancelled orders (sales should be 0, profit should be negative):')
print(export_df[export_df['order_status'] == 'CANCELED'][['sales', 'order_profit_per_order', 'order_item_total']].head(3))

In [ ]:
import os

# Save to CSV
output_file = 'dataco_powerbi_master.csv'
export_df.to_csv(output_file, index=False)

# Confirm the file was saved correctly
file_size = os.path.getsize(output_file) / (1024 * 1024)
check = pd.read_csv(output_file, nrows=3)

print(f'File saved: {output_file}')
print(f'Rows: {len(export_df):,}')
print(f'Columns: {len(export_df.columns)}')
print(f'File size: {file_size:.1f} MB')
print(f'Location: {os.path.abspath(output_file)}')
print('\nFirst 3 rows preview:')
check

---
## Section 8 — Key Findings Summary

Here is a summary of the most important things we found in this analysis.

| Finding | Number | What it means |
|---|---|---|
| True late delivery rate | **54.8%** | More than half of all orders arrive late. The system was reporting ~40% because 4,423 risk flags were wrong. |
| Revenue at risk | **$20.6M** | This much revenue is tied to orders that arrived late. |
| Worst shipping mode | **First Class — 95% late** | First Class is the most expensive option but has the worst delivery performance. |
| Total revenue | **$32.39M** | Net revenue after discounts across all 5 markets. |
| Total profit | **$3.83M** | After all costs, this is what the company keeps (11.8% margin). |
| Total discounts given | **$3.65M** | Roughly equal to the total profit — too much discount is being given. |
| Best market | **Europe ($9.6M)** | Europe is the top market by revenue. Africa is the smallest at $2.0M. |
| Top category | **Fishing** | Fishing earns the most revenue ($6.1M) and the highest profit ($0.73M). |
| All fraud via Transfer | **100%** | Every single fraud order was paid using the Transfer payment method. |
| Fraud concentration | **Caguas city** | Over 1,500 of the 4,062 fraud cases came from one city. |
| Best profit segment | **Corporate** | Corporate customers have a 32% profit ratio vs 23% for Consumer. |
| Weakest quarter | **Q4** | October, November and December have lower order volumes. |
| LATAM pattern | **Q1 and Q2 only** | LATAM market generates almost no orders in the second half of the year. |